In [8]:
import os
import pickle
import nltk
from transformers import BertTokenizer

nltk.download('punkt')
nltk.download('punkt_tab') # Added to download the missing resource

# Initialize tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


### Helper Function

In [2]:
def align_and_tokenize(sentences, tags, strategy):
    processed_sents = []
    processed_tags = []

    tag_map = {'O': 0, 'B-Disease': 1, 'I-Disease': 2, 'B-Chemical': 3, 'I-Chemical': 4}

    for tokens, labels in zip(sentences, tags):
        sentence_str = " ".join(tokens)

        if strategy == 'whitespace':
            new_tokens = sentence_str.split()
            # Map labels directly
            new_labels = [tag_map.get(l, 0) for l in labels[:len(new_tokens)]]

        elif strategy == 'nltk':
            new_tokens = nltk.word_tokenize(sentence_str)
            # Simple fallback alignment matching original lengths
            new_labels = []
            for i, tok in enumerate(new_tokens):
                orig_idx = min(i, len(labels) - 1)
                new_labels.append(tag_map.get(labels[orig_idx], 0))

        elif strategy == 'wordpiece':
            new_tokens = []
            new_labels = []
            for word, label in zip(tokens, labels):
                sub_tokens = bert_tokenizer.tokenize(word)
                if not sub_tokens:
                    continue
                new_tokens.extend(sub_tokens)
                # Assign tag to first subword, ignore index (-100) for subsequent subwords
                new_labels.append(tag_map.get(label, 0))
                new_labels.extend([-100] * (len(sub_tokens) - 1))

        processed_sents.append(new_tokens)
        processed_tags.append(new_labels)

    return processed_sents, processed_tags

In [3]:
# Create output folder
os.makedirs('/content/drive/MyDrive/embeddings_assignment/processed_data', exist_ok=True)

splits = ['train', 'val', 'test']
strategies = ['whitespace', 'nltk', 'wordpiece']

In [5]:
def conll_reader(file_path):
    sentences = []
    tags = []

    current_sentence = []
    current_tags = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Sentence boundary
            if line == "":
                if current_sentence:
                    sentences.append(current_sentence)
                    tags.append(current_tags)

                    current_sentence = []
                    current_tags = []
                continue

            parts = line.split()

            # Assumes: Token Tag
            token = parts[0]
            tag = parts[-1]

            current_sentence.append(token)
            current_tags.append(tag)

    # Handle last sentence if file doesn't end with blank line
    if current_sentence:
        sentences.append(current_sentence)
        tags.append(current_tags)

    return sentences, tags

### Perform all Tockenizers

In [9]:
paths = {
    "train": "/content/drive/MyDrive/embeddings_assignment/train.txt",
    "val": "/content/drive/MyDrive/embeddings_assignment/val.txt",
    "test": "/content/drive/MyDrive/embeddings_assignment/test.txt"
}

for split in splits:
    raw_sents, raw_tags = conll_reader(paths[split])

    for strategy in strategies:
        sents, labels = align_and_tokenize(raw_sents, raw_tags, strategy)

        output_file = f"/content/drive/MyDrive/embeddings_assignment/processed_data/{split}_{strategy}.pkl"

        with open(output_file, "wb") as f:
            pickle.dump(
                {
                    "tokens": sents,
                    "tags": labels
                },
                f
            )

        print(f"Saved: {output_file}")

print("Preprocessing fully complete!")

Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/train_whitespace.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/train_nltk.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/train_wordpiece.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/val_whitespace.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/val_nltk.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/val_wordpiece.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/test_whitespace.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/test_nltk.pkl
Saved: /content/drive/MyDrive/embeddings_assignment/processed_data/test_wordpiece.pkl
Preprocessing fully complete!
